# 🧠 Mental Health Intelligence Platform

## Phase 5 — Feature Engineering

### Overview

Feature engineering transforms raw survey responses into machine learning-ready features.

Well-designed features improve predictive performance while increasing the interpretability of machine learning models.

This notebook prepares the harmonized dataset for predictive modeling.

---

### Objectives

The objectives of this notebook are:

- Select relevant predictive variables
- Encode categorical variables
- Create engineered features
- Prepare the target variable
- Build a machine learning-ready dataset

## Import Libraries

In [14]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

## Load Dataset

In [15]:
df = pd.read_csv("../data/processed/harmonized_osmi_data.csv")

print(df.shape)

df.head()

(3082, 70)


,source_file,survey_year,self_employed,no_employees,tech_company,primary_role_tech,benefits,know_options,formal_discussion,resources,...,improvement_suggestions,additional_comments,willing_to_interview,age,gender,country_live,state_live,race,country_work,state_work
0,2014,2014,NaN,2025-06-01 00:00:00,Yes,NaN,Yes,Don't Know,No,Yes,...,NaN,NaN,NaN,NaN,Male,NaN,NaN,NaN,NaN,NaN
1,2014,2014,NaN,More than 1000,No,NaN,Don't Know,No,Don't Know,Don't Know,...,NaN,NaN,NaN,NaN,Other,NaN,NaN,NaN,NaN,NaN
2,2014,2014,NaN,2025-06-01 00:00:00,Yes,NaN,No,No,No,No,...,NaN,NaN,NaN,NaN,Other,NaN,NaN,NaN,NaN,NaN
3,2014,2014,NaN,26-100,Yes,NaN,No,Yes,No,No,...,NaN,NaN,NaN,NaN,Other,NaN,NaN,NaN,NaN,NaN
4,2014,2014,NaN,100-500,Yes,NaN,Yes,No,Don't Know,Don't Know,...,NaN,NaN,NaN,NaN,Male,NaN,NaN,NaN,NaN,NaN


## Remove Free-Text Columns

In [16]:
text_cols = [
    "unsupportive_response_desc",
    "identified_affected_how",
    "bring_up_mental_why",
    "improvement_suggestions",
    "additional_comments"
]

cols = [c for c in text_cols if c in df.columns]

df = df.drop(columns=cols)

print(f"Removed {len(cols)} free-text columns.")

Removed 5 free-text columns.


## Fix Company Size Categories

In [19]:
company_map = {
    "2025-06-01 00:00:00": "6-25",
    "2026-05-01 00:00:00": "51-100"
}

if "no_employees" in df.columns:
    df["no_employees"] = df["no_employees"].replace(company_map)

## Fix Leave Categories

In [20]:
if "leave_ease" in df.columns:

    df["leave_ease"] = (
        df["leave_ease"]
        .replace(
            {
                "Very Difficult":"Very difficult",
                "Very Easy":"Very easy"
            }
        )
    )

## Group Rare Countries

In [21]:
country_counts = df["country_live"].value_counts()

rare = country_counts[country_counts < 20].index

df["country_live"] = (
    df["country_live"]
    .replace(rare, "Other")
)

## Group Rare States

In [22]:
if "state_live" in df.columns:

    state_counts = df["state_live"].value_counts()

    rare = state_counts[state_counts < 20].index

    df["state_live"] = (
        df["state_live"]
        .replace(rare, "Other")
    )

## Group Rare Races

In [23]:
if "race" in df.columns:

    race_counts = df["race"].value_counts()

    rare = race_counts[race_counts < 20].index

    df["race"] = (
        df["race"]
        .replace(rare, "Other")
    )

## Create Age Groups

In [24]:
df["age_group"] = pd.cut(

    df["age"],

    bins=[18,25,35,45,55,70],

    labels=[

        "18-24",

        "25-34",

        "35-44",

        "45-54",

        "55+"

    ]
)

## Workplace Support Score

In [26]:
support_cols = [
    "benefits",
    "resources",
    "formal_discussion",
    "anonymity"
]

binary_map = {
    "Yes": 1,
    "No": 0
}

# Create a copy of the subset for safety
temp = df[support_cols].copy()

for col in support_cols:
    # First, convert any non-string values to string representation
    temp[col] = temp[col].astype(str).str.strip()
    # Map 'Yes' to 1, everything else to 0
    temp[col] = temp[col].apply(lambda x: 1 if x == 'Yes' else 0)
    
df["workplace_support_score"] = temp.sum(axis=1)

# Verify the results
print(f"Support score range: {df['workplace_support_score'].min()} to {df['workplace_support_score'].max()}")
print(f"Mean support score: {df['workplace_support_score'].mean():.2f}")
print(f"Distribution:\n{df['workplace_support_score'].value_counts().sort_index()}")

Support score range: 0 to 4
Mean support score: 1.09
Distribution:
workplace_support_score
0    1450
1     666
2     430
3     326
4     210
Name: count, dtype: int64


## Missing Value Summary

In [27]:
summary = pd.DataFrame({

    "Missing":df.isna().sum(),

    "Percent":round(df.isna().mean()*100,2)

})

summary = summary[summary["Missing"]>0]

summary.sort_values("Percent",ascending=False)

,Missing,Percent
productivity_percent_affected,2888,93.71
identified_affected_career,2860,92.80
reveal_to_clients,2823,91.60
know_resources,2823,91.60
coverage_mental_health,2823,91.60
...,...,...
anonymity,259,8.40
leave_ease,259,8.40
comfort_physical_mental,259,8.40
comfort_supervisor,259,8.40


## Save Engineered Dataset

In [28]:
df.to_csv(

    "../data/processed/engineered_osmi_data.csv",

    index=False

)

print("Feature engineered dataset saved.")

Feature engineered dataset saved.
